# 📓 실습 02. Dense 유사도와 그래프 구조 스코어를 결합한 하이브리드 리트리버

이 실습에서는 사용자 질문 쿼리가 들어왔을 때, 텍스트 청크 간의 통계 유사도(유사 임베딩 코사인 유사도)와 그래프 구조상의 최단 노드 위상 거리(BFS)를 융합하여 정밀 탐색하는 **가중 융합 하이브리드 검색 알고리즘**을 구현합니다.

In [ ]:
import os
import json
import numpy as np
import networkx as nx

# 1. 데이터 로드
chunks_path = os.path.join("..", "01_structural_ingest_parsing", "data", "parsed_chunks.json")
with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)
    
triplets_path = os.path.join("..", "02_knowledge_triplet_extraction", "data", "extracted_triplets.json")
with open(triplets_path, "r", encoding="utf-8") as f:
    triplets = json.load(f)

In [ ]:
# 2. 코사인 유사도 및 그래프 BFS 위상 스코어 알고리즘 구현
def get_mock_vector_similarity(query, chunks):
    # 실제 임베딩 모델 대신 쿼리 단어 출현 빈도로 모의 유사도 리턴
    scores = []
    query_words = set(query.split())
    for chunk in chunks:
        content = chunk["content"]
        match_count = sum(1 for word in query_words if word in content)
        scores.append(float(match_count / max(1, len(query_words))))
    return np.array(scores)

def get_graph_topological_similarity(query, triplets, chunks):
    # 지식 그래프 생성
    G = nx.Graph()
    for t in triplets:
        G.add_edge(t["subject"], t["object"])
        
    # 쿼리와 관계된 엔티티 탐색
    target_nodes = [node for node in G.nodes if any(word in node for word in query.split())]
    
    scores = []
    for idx, chunk in enumerate(chunks):
        content = chunk["content"]
        chunk_ents = [node for node in G.nodes if node in content]
        
        # 최단 경로 탐색 스코어
        min_dist = float('inf')
        for start in target_nodes:
            for end in chunk_ents:
                try:
                    dist = nx.shortest_path_length(G, source=start, target=end)
                    if dist < min_dist:
                        min_dist = dist
                except:
                    pass
        
        # 거리가 가까울 수록 높은 위상 스코어 부여
        if min_dist == float('inf'):
            scores.append(0.0)
        else:
            scores.append(1.0 / (min_dist + 1))
            
    return np.array(scores)

# 3. 하이브리드 가중 융합 검색 구동
def hybrid_retrieve(query, chunks, triplets, alpha=0.6):
    s_vec = get_mock_vector_similarity(query, chunks)
    s_grp = get_graph_topological_similarity(query, triplets, chunks)
    
    # 가중치 결합
    s_total = alpha * s_vec + (1 - alpha) * s_grp
    best_idx = np.argmax(s_total)
    
    return chunks[best_idx], s_total[best_idx]

query = "슬러리 노즐 막힘 장애 발생 시 정비 엔지니어의 SOP 조치"
best_chunk, score = hybrid_retrieve(query, chunks, triplets, alpha=0.5)

print(f"질의: {query}\n")
print(f"최적 매칭 스코어: {score:.4f}")
print(f"매칭 계층: {best_chunk['headers']}")
print(f"검색된 조치 가이드 내용:\n{best_chunk['content']}")